In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import os
import numpy as np

In [2]:
# List files in the broadcast_logs directory
os.listdir("../../data/broadcast_logs/")

['BroadcastLogs_2018_Q3_M8_sample.CSV',
 'data dictionary.doc',
 '.DS_Store',
 'Call_Signs.csv',
 'ReferenceTables']

In [3]:
# Create a Spark session
spark = SparkSession.builder.appName("Ch04").getOrCreate()

# Define the directory containing the data
DIRECTORY = "../../data/broadcast_logs/"

# Read the CSV file into a DataFrame
logs = spark.read.csv(
    os.path.join(DIRECTORY, "BroadcastLogs_2018_Q3_M8_sample.CSV"),
    sep = "|",
    header=True,
    inferSchema=True,
    timestampFormat="yyyy-MM-dd"
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/31 14:14:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# Print the schema of the DataFrame
logs.printSchema()

root
 |-- BroadcastLogID: integer (nullable = true)
 |-- LogServiceID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- SequenceNO: integer (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- ProgramClassID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable 

In [5]:
# Split the DataFrame columns into groups of ten and display the first five rows of each group
column_split = np.array_split(logs.columns, len(logs.columns) // 10)

for x in column_split:
    logs.select(*x).show(5, False)

+--------------+------------+----------+----------+-------------------+----------------------+----------+---------------+-----------------+----------------+
|BroadcastLogID|LogServiceID|LogDate   |SequenceNO|AudienceTargetAgeID|AudienceTargetEthnicID|CategoryID|ClosedCaptionID|CountryOfOriginID|DubDramaCreditID|
+--------------+------------+----------+----------+-------------------+----------------------+----------+---------------+-----------------+----------------+
|1196192316    |3157        |2018-08-01|1         |4                  |NULL                  |13        |3              |3                |NULL            |
|1196192317    |3157        |2018-08-01|2         |NULL               |NULL                  |NULL      |1              |NULL             |NULL            |
|1196192318    |3157        |2018-08-01|3         |NULL               |NULL                  |NULL      |1              |NULL             |NULL            |
|1196192319    |3157        |2018-08-01|4         |NULL   

In [6]:
# drop unnecessary columns
logs = logs.drop("BroadcastLogID", "SequenceNO")

In [7]:
# Extract hours, minutes, and seconds from the Duration column
logs.select(F.col("Duration"),
            F.col("Duration").substr(1,2).cast("int").alias("dur_hours"),
            F.col("Duration").substr(4,2).cast("int").alias("dur_minutes"),
            F.col("Duration").substr(7,2).cast("int").alias("dur_seconds"),
).distinct().show(5, False)
            

+----------------+---------+-----------+-----------+
|Duration        |dur_hours|dur_minutes|dur_seconds|
+----------------+---------+-----------+-----------+
|00:04:52.0000000|0        |4          |52         |
|00:10:06.0000000|0        |10         |6          |
|00:26:41.0000000|0        |26         |41         |
|00:09:52.0000000|0        |9          |52         |
|00:04:26.0000000|0        |4          |26         |
+----------------+---------+-----------+-----------+
only showing top 5 rows



In [8]:
# Create a new column "Duration_seconds" that converts the Duration into total seconds
logs = logs.withColumn(
    "Duration_seconds",
    (
        F.col("Duration").substr(1,2).cast("int") * 3600 +
        F.col("Duration").substr(4,2).cast("int") * 60 +
        F.col("Duration").substr(7,2).cast("int")
    ),
)

In [9]:
# Display summary statistics for the Duration_seconds column
logs.describe("Duration_seconds").show()

+-------+------------------+
|summary|  Duration_seconds|
+-------+------------------+
|  count|            236724|
|   mean|124.30587942076004|
| stddev| 573.7742807594924|
|    min|                 1|
|    max|             23409|
+-------+------------------+



In [10]:
# Alternative method to display summary statistics
logs.select("Duration_seconds").summary().show()

+-------+------------------+
|summary|  Duration_seconds|
+-------+------------------+
|  count|            236724|
|   mean|124.30587942076004|
| stddev| 573.7742807594924|
|    min|                 1|
|    25%|                15|
|    50%|                15|
|    75%|                30|
|    max|             23409|
+-------+------------------+



In [11]:
# Detailed summary statistics including specific percentiles
logs.select("Duration_seconds").summary("min", "10%", "50%", "90%", "max").show()

+-------+----------------+
|summary|Duration_seconds|
+-------+----------------+
|    min|               1|
|    10%|              15|
|    50%|              15|
|    90%|              30|
|    max|           23409|
+-------+----------------+



In [12]:
# List files in the ReferenceTables directory
os.listdir("../../data/broadcast_logs/ReferenceTables")

['CD_BroadcastOriginPoint.csv',
 'CD_ProgramClass.csv',
 'CD_Composition.csv',
 'CD_Exhibition.csv',
 'CD_Category.csv',
 'CD_AudienceTargetAge.csv',
 'CD_NetworkAffiliation.csv',
 'CD_ProductionSource.csv',
 'CD_DubDramaCredit.csv',
 'CD_AirLanguage.csv',
 'CD_SpecialAttention.csv',
 'CD_ClosedCaption.csv',
 'BroadcastProducers.csv',
 'CD_CountryOfOrigin.csv',
 'CD_EthnicProgram.csv',
 'CD_FilmClassification.csv',
 'LogIdentifier.csv',
 'CD_AudienceTargetEthnic.csv']

In [13]:
# Read the CSV file into a DataFrame
log_identifier = spark.read.csv(
    os.path.join(DIRECTORY, "ReferenceTables/LogIdentifier.csv"),
    sep = "|",
    header=True,
    inferSchema=True,
)
log_identifier.show(5, False)

+---------------+------------+---------+
|LogIdentifierID|LogServiceID|PrimaryFG|
+---------------+------------+---------+
|13ST           |3157        |1        |
|2000SM         |3466        |1        |
|70SM           |3883        |1        |
|80SM           |3590        |1        |
|90SM           |3470        |1        |
+---------------+------------+---------+
only showing top 5 rows



In [14]:
log_identifier = log_identifier.where(F.col("PrimaryFG") == 1)
print(f"Number of unique LogIdentifiers with PrimaryFG=1: {log_identifier.count()}")

Number of unique LogIdentifiers with PrimaryFG=1: 758


In [15]:
logs_and_channels = logs.join(log_identifier, "LogServiceID")
logs_and_channels.printSchema()

root
 |-- LogServiceID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- ProgramClassID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable = true)
 |-- NetworkAffiliationID: integer (nullable = true)
 |-- SpecialAttentionID: inte

In [16]:
# Read the CD_Category CSV file into a DataFrame
cd_category = spark.read.csv(
    os.path.join(DIRECTORY, "ReferenceTables/CD_Category.csv"),
    sep = "|",
    header=True,
    inferSchema=True,
).select(
    "CategoryID", 
    "CategoryCD",
    F.col("EnglishDescription").alias("Category_Description"),
)
cd_category.show(5, False)

+----------+----------+---------------------------+
|CategoryID|CategoryCD|Category_Description       |
+----------+----------+---------------------------+
|1         |010       |NEWS                       |
|2         |02        |CANREC  ANALYSIS (old)     |
|3         |02A       |ANALYSIS AND INTERPRETATION|
|4         |02B       |LONG-FORM DOCUMENTARY      |
|5         |030       |REPORTING & ACTUALITIES    |
+----------+----------+---------------------------+
only showing top 5 rows



In [17]:
# Read the CD_ProgramClass CSV file into a DataFrame
cd_program_class = spark.read.csv(
    os.path.join(DIRECTORY, "ReferenceTables/CD_ProgramClass.csv"),
    sep = "|",
    header=True,
    inferSchema=True,
).select(
    "ProgramClassID", 
    "ProgramClassCD",
    F.col("EnglishDescription").alias("ProgramClass_Description"),
)
cd_program_class.show(5, False)

+--------------+--------------+------------------------+
|ProgramClassID|ProgramClassCD|ProgramClass_Description|
+--------------+--------------+------------------------+
|1             |AUT           |AUTOPROMOTION           |
|2             |BAL           |BALANCE PROGRAMMING     |
|3             |COM           |COMMERCIAL MESSAGE      |
|4             |COR           |CORNERSTONE             |
|5             |DOC           |DOCUMENTARY             |
+--------------+--------------+------------------------+
only showing top 5 rows



In [18]:
# Join logs_and_channels with cd_category and cd_program_class to create full_logs
full_logs = logs_and_channels.join(cd_category, "CategoryID", how = "left").join(cd_program_class, "ProgramClassID", how = "left")
full_logs.printSchema()

root
 |-- ProgramClassID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- LogServiceID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable = true)
 |-- NetworkAffiliationID: integer (nullable = true)
 |-- SpecialAttentionID: inte

In [19]:
# Display the most popular program classes by total duration
full_logs.groupBy("ProgramClassCD", "ProgramClass_Description") \
    .agg(F.sum("Duration_seconds").alias("Total_Duration_Seconds")) \
    .orderBy(F.desc("Total_Duration_Seconds")) \
    .show(10, False)

+--------------+--------------------------------------+----------------------+
|ProgramClassCD|ProgramClass_Description              |Total_Duration_Seconds|
+--------------+--------------------------------------+----------------------+
|PGR           |PROGRAM                               |20992510              |
|COM           |COMMERCIAL MESSAGE                    |3519163               |
|PFS           |PROGRAM FIRST SEGMENT                 |1344762               |
|SEG           |SEGMENT OF A PROGRAM                  |1205998               |
|PRC           |PROMOTION OF UPCOMING CANADIAN PROGRAM|880600                |
|PGI           |PROGRAM INFOMERCIAL                   |679182                |
|PRO           |PROMOTION OF NON-CANADIAN PROGRAM     |335701                |
|OFF           |SCHEDULED OFF AIR TIME PERIOD         |142279                |
|ID            |NETWORK IDENTIFICATION MESSAGE        |74926                 |
|NRN           |No recognized nationality           

In [20]:
# Calculate the percentage of commercial programming for each LogIdentifierID
answer = (
    full_logs.groupBy("LogIdentifierID")
    .agg(
        F.sum(
            F.when(
                F.trim(F.col("ProgramClassCD")).isin(["COM", "PRC", "PGI", "PRO", "LOC", "SPO", "MER", "SOL"]),
                F.col("Duration_seconds")
                ).otherwise(0)
            ).alias("Total_Commercial_Duration_Seconds"),
        F.sum("Duration_seconds").alias("Total_Duration_Seconds"),
        )
    .withColumn(
        "Commercial_Percentage",
        F.col("Total_Commercial_Duration_Seconds") / F.col("Total_Duration_Seconds") * 100
    )
)


In [ ]:
# Display the top 10 LogIdentifierIDs with the highest commercial percentage
answer.orderBy(F.desc("Commercial_Percentage")).show(10, False)

+---------------+---------------------------------+----------------------+---------------------+
|LogIdentifierID|Total_Commercial_Duration_Seconds|Total_Duration_Seconds|Commercial_Percentage|
+---------------+---------------------------------+----------------------+---------------------+
|CIMT           |775                              |775                   |100.0                |
|MSET           |2700                             |2700                  |100.0                |
|TLNSP          |15480                            |15480                 |100.0                |
|TELENO         |17790                            |17790                 |100.0                |
|HPITV          |13                               |13                    |100.0                |
|TANG           |8125                             |8125                  |100.0                |
|MMAX           |23333                            |23582                 |98.94410991434145    |
|MPLU           |20587        

In [ ]:
# Display the top 10 LogIdentifierIDs with the lowest commercial percentage
answer.orderBy(F.asc("Commercial_Percentage")).show(10, False)

+---------------+---------------------------------+----------------------+---------------------+
|LogIdentifierID|Total_Commercial_Duration_Seconds|Total_Duration_Seconds|Commercial_Percentage|
+---------------+---------------------------------+----------------------+---------------------+
|EURO           |0                                |NULL                  |NULL                 |
|NINOS          |0                                |NULL                  |NULL                 |
|PLAY           |0                                |86400                 |0.0                  |
|CFTV           |0                                |102                   |0.0                  |
|CFTF           |0                                |1805                  |0.0                  |
|CKRT           |0                                |14400                 |0.0                  |
|SKIN           |0                                |86400                 |0.0                  |
|SNONE          |0            

In [21]:
# Handle null values in the answer DataFrame
answer_no_null = answer.fillna(0)